# 00 · Environment and provenance

This suite validates that OpenFF Interchange exports SMIRNOFF **virtual sites**
to OpenMM correctly, with and without going through `openmmforcefields`'
`SMIRNOFFTemplateGenerator`.

This notebook pins and records the exact software versions the results are
certified against, and documents the specific Interchange vsite bugs the suite
regression-locks.

In [1]:
import platform
import importlib.metadata as md

import openmm

print("Python :", platform.python_version())
print("Platform:", platform.platform())
print()
PKGS = [
    "openff-toolkit", "openff-interchange", "openff-units", "openff-utilities",
    "openmm", "openmmforcefields", "openforcefields", "openff-nagl",
    "openff-nagl-models", "mdanalysis", "numpy",
]
versions = {}
for p in PKGS:
    try:
        versions[p] = md.version(p)
    except Exception:
        versions[p] = "MISSING"
    print(f"  {p:22s} {versions[p]}")

print("\nOpenMM plugins:", ", ".join(openmm.pluginLoadedLibNames) or "(none)")

Python : 3.12.13
Platform: macOS-15.6.1-arm64-arm-64bit

  openff-toolkit         0.18.1
  openff-interchange     0.5.4.dev4+g7e742e906
  openff-units           0.3.2
  openff-utilities       0.1.16
  openmm                 8.5.2
  openmmforcefields      0.16.0
  openforcefields        2026.1.0
  openff-nagl            0.0.0
  openff-nagl-models     2025.9.0
  mdanalysis             2.10.0
  numpy                  1.26.4

OpenMM plugins: /Users/jeffreywagner/micromamba/envs/vsite-validation/lib/plugins/libOpenMMPME.dylib, /Users/jeffreywagner/micromamba/envs/vsite-validation/lib/plugins/libOpenMMCPU.dylib, /Users/jeffreywagner/micromamba/envs/vsite-validation/lib/plugins/libOpenMMOpenCL.dylib, /Users/jeffreywagner/micromamba/envs/vsite-validation/lib/plugins/libOpenMMRPMDOpenCL.dylib, /Users/jeffreywagner/micromamba/envs/vsite-validation/lib/plugins/libOpenMMDrudeOpenCL.dylib, /Users/jeffreywagner/micromamba/envs/vsite-validation/lib/plugins/libOpenMMAmoebaOpenCL.dylib, /Users/jeffreyw

## Minimum-version gates

Most virtual-site fixes this suite depends on landed in **Interchange 0.5.1**
(see the bug table below). The last one, **#1498**, was merged *after* the 0.5.3
release, so this suite runs against a patched Interchange installed from the
`main` branch (version `> 0.5.3`, e.g. `0.5.4.dev*`), and *verifies* that fix
rather than documenting it as an outstanding bug. `openmmforcefields` gained
SMIRNOFF virtual-site + multi-residue support in **0.16.0** (PRs #423, #430). We
assert these minimums so a stale environment fails loudly rather than silently
reporting "PASS".

In [2]:
from packaging.version import Version


def _ge(pkg, minimum):
    v = versions[pkg]
    assert v != "MISSING", f"{pkg} is not installed"
    assert Version(v) >= Version(minimum), f"{pkg} {v} < required {minimum}"
    print(f"  OK  {pkg} {v} >= {minimum}")


# Interchange must be newer than 0.5.3 so the #1498 fix is present.
_ic = Version(versions["openff-interchange"])
assert _ic > Version("0.5.3"), (
    f"openff-interchange {_ic} predates the #1498 fix; install from main "
    "(pip install 'git+https://github.com/openforcefield/openff-interchange.git@main').")
print(f"  OK  openff-interchange {_ic} > 0.5.3 (includes #1498 fix)")
_ge("openmmforcefields", "0.16.0")
_ge("openmm", "8.5.0")
_ge("openff-toolkit", "0.18.0")
print("\nEnvironment satisfies minimum versions.")

  OK  openff-interchange 0.5.4.dev4+g7e742e906 > 0.5.3 (includes #1498 fix)
  OK  openmmforcefields 0.16.0 >= 0.16.0
  OK  openmm 8.5.2 >= 8.5.0
  OK  openff-toolkit 0.18.1 >= 0.18.0

Environment satisfies minimum versions.


## Virtual-site bugs regression-locked by this suite

All surfaced in early 2026 while cross-checking Interchange against
`openmmforcefields`, and fixed in Interchange 0.5.1 unless noted. Each is
exercised by the notebook listed.

| Issue | Summary | Fixed in | Exercised by |
|-------|---------|----------|--------------|
| interchange&nbsp;#1436 | Non-default 1-4 scaling not applied to vsite–vsite / vsite–atom exceptions | 0.5.1 | `03` (and `01`) |
| interchange&nbsp;#1438 | `DivalentLonePair` geometry wrong | 0.5.1 | `03` |
| interchange&nbsp;#939  | `MonovalentLonePair` position off | (tracked) | `03` |
| interchange&nbsp;#1498 | vsite exceptions zeroed when `combine_nonbonded_forces=False` | after 0.5.3 (main) | `01` (fix verified) |
| hybrid-topology copy | `LocalCoordinatesSite` vsites in a hybrid-topology free-energy System | — | `05` (deferred design) |

The core representational fact under test: **Interchange exports every SMIRNOFF
vsite as an OpenMM `LocalCoordinatesSite`**, whereas `openmmforcefields`' bundled
water models use `ThreeParticleAverageSite`/`OutOfPlaneSite`. Given identical
real-atom coordinates the two encodings must place the vsite identically and
yield identical energies — notebook `02` checks exactly this.

In [3]:
# Record provenance to a JSON file so downstream notebooks / readers can cite it.
import json
import datetime

provenance = {
    "generated": datetime.datetime.now().isoformat(timespec="seconds"),
    "python": platform.python_version(),
    "platform": platform.platform(),
    "versions": versions,
}
with open("provenance.json", "w") as fh:
    json.dump(provenance, fh, indent=2)
print("Wrote provenance.json")

Wrote provenance.json
